# 分類性能と公平性 — ResNet vs attribute-invariant ResNet

通常 ResNet と attribute-invariant ResNet の best validation-AUROC checkpoint を、同じ test split で
比べる。**invariant 化で分類性能をどれだけ払い、その見返りに属性間の差がどう動いたか**を、
2 つの表として並べる。片方だけを見ると「gap は縮んだが全体も落ちた」を見落とす。

集計は `groups.py` が持つ。この notebook は表の表示と読み取りだけを行う。

```bash
uv run python analysis/common/predictions.py --study initial-resnet-vs-invariant --split test \
  --run-dir projects/hypernet_e2e/runs/20260921T103036Z-resnet-chexpert-s42-5538 \
  --run-dir projects/hypernet_e2e/runs/20260921T125711Z-resnet-chexpert-attribute-invariant-s42-9fd3
uv run python analysis/initial-resnet-vs-invariant/groups.py --split test
```

In [ ]:
from pathlib import Path

import pandas as pd
import rootutils

ROOT = rootutils.setup_root(Path.cwd(), indicator=".project-root", pythonpath=True)
RESULTS = ROOT / "analysis" / "initial-resnet-vs-invariant" / "results"
SPLIT = "test"

## 全体性能

`cross_entropy` も見る。AUROC が同じでも確率の較正が崩れていることがあるため。

In [ ]:
pd.read_csv(RESULTS / f"classification_performance_{SPLIT}.csv").round(4)

## 属性ごとの公平性

学習側と同じ `compute_fairness_metrics` の定義で、sex / race / ethnicity / age group (65 歳) を見る。
`Eopp1` は群間の TPR の最大差、`Eopp0` は TNR の最大差、`Eodds` はその平均。worst-group の値も
同じ表にあるので、gap の縮みが worst の改善なのか best の低下なのかを 1 行の中で読める。

In [ ]:
fairness = pd.read_csv(RESULTS / f"fairness_metrics_{SPLIT}.csv")
fairness.set_index(["attribute", "model"]).drop(columns="split").round(4)

---

**この比較は seed 42 の各 1 run に限る。** 公平性の差を主張する前に、複数 seed で再検証する。
交差群まで見る場合は、同じ予測 cache から [iterative-probe](../iterative-probe/) の `groups.py` と
同じ切り方で群を作る。